# agent_obs_alert

## In plain terms
Think of this notebook as the control room. Every other notebook in this pipeline watches for
one kind of problem; this one collects everything they find, writes it to a running log
(`obs_incidents`) so nothing gets lost or double-counted, and tells a human about it — once per
problem, not once every few minutes. If the monitoring itself breaks (a table goes missing, a
query errors out), it fails loudly instead of quietly going dark. Every detector that's currently
live is treated as equally serious (CRITICAL) — there's no "just a warning" tier in production
right now.

If you got paged by this: the Teams message includes what fired, why it matters, and a
ready-to-run query to go look at the underlying data yourself (see "Where to look" in the card).

Central alerting notebook for the ISH/SAA agent observability pipeline. This notebook:
1. persists findings to `obs_incidents` (append-only, MERGE-deduped),
2. evaluates each detector condition in Python,
3. bounds every check in time,
4. reports missing/stale upstream tables as `UNAVAILABLE` rather than crashing,
5. posts a triage-oriented Adaptive Card to Teams — once per incident, not once per run,
6. raises when a detector is broken, so Databricks' own job notifications fire.

## 16 active detectors (prod v1) — all CRITICAL, no WARN-status detectors remain
Every detector below either persists to `obs_incidents` and notifies Teams, or has been
retired outright (2026-09-21 detector value audit) — see "Retired detectors" below. No
detector ships WARN-only/observe-forever any more. (Count = 11 in `INCIDENT_SOURCES` + 5 in
`SCALAR_INCIDENT_SOURCES` in cell 5 — verified 2026-09-21 against the code; this header
previously undercounted at 15 by bundling #8/#9 below into one bullet.)

1. **Handover delivery failures** (CRITICAL) — email send failures from ISH audit
2. **Handover delivery rate** (CRITICAL) — 7-day rolling failure rate >20%, OR failed>=3
   absolute-count floor (added 2026-09-21 — see #4 below) regardless of sample size
3. **Shift context missing** (CRITICAL, added 2026-09-18 as WARN; promoted 2026-09-21) —
   blank shift_date/shift_type/batch_nbr in output records. Promoted because a standing
   data-quality gap that silently degrades every AI-to-ISH correlation join was judged not
   worth leaving invisible to a human indefinitely — promote-on-principle, not a fresh
   empirical FP re-validation. See `ptof_obs_liveness_detection.ipynb`.
4. **Blank output** (CRITICAL) — content is NULL/empty/trivial on an output record. Fires on
   any blank output at any volume (2026-09-21, FP/FN bias review step 3, Option A signed off) —
   the prior count/rate/volume floor was dropped entirely; see `ptof_obs_mal_output.ipynb`.
5. **Schema field missing** (CRITICAL) — a JSON key that used to reliably appear has
   disappeared vs baseline. (The field_added half of this comparison was retired 2026-09-21 —
   see "Retired detectors" below.)
6. **Capability silence** (CRITICAL, added as WARN; promoted 2026-09-21, evidence-based) —
   output_type hasn't generated in >grace_hours. Promoted on real historical evidence: full
   call-gap distribution for all 3 covered capabilities shows 0 empirical false positives ever,
   with real margin over the grace threshold (saa-display/situational-awareness 7.4x,
   summary 3.0x).
7. **Pipeline heartbeat** (CRITICAL) — zero outputs from any capability in 25 minutes
   (tightened from 2h -> 45min -> 25min 2026-09-21 — real max gap over 30 days is 16.4 min)
8. **ETL pipeline failure** (CRITICAL) — an ETL source table refresh outright failed.
9. **ETL pipeline staleness** (CRITICAL) — the global ETL fleet (max across all tables) hasn't
   completed a run in >30min. Distinct from #8 (a run recorded as failed) — this fires on the
   absence of any run at all. Split out 2026-09-21 into its own bullet from a prior "ETL
   pipeline health" bullet that bundled it with #8 — same two detectors, code unchanged, just
   corrected the header to count them separately (they were already two separate check() calls
   and two separate detector names in the code).
10. **ETL table staleness** (CRITICAL, added 2026-09-21) — a single ETL source table hasn't
    completed a run in >60 min, independent of whether the rest of the fleet is still running.
    Per-table companion to #9's global `etl_pipeline_staleness` scalar — closes a confirmed
    masking gap where a partial ETL stall was invisible to the global check because unaffected
    tables kept its `max(run_timestamp)` fresh. Runs alongside #9, not instead of it.
11. **ETL run slow** (CRITICAL, added as WARN; promoted 2026-09-21) — ETL run duration
    degraded beyond its own MAD-based historical bound, clustered (>=3 in an hour), rolled up
    to `(task_name, window_start)`. Promoted after 5 real, correlated, multi-table slowdown
    events were observed in ~1 week while it was WARN/log-only and invisible to a human.
    Per-incident Teams notification suppressed in favor of a daily digest (2026-09-21, FP/FN
    bias review step 3, signed off "should be changed to a digest") — the same signal that
    justified promoting it also makes a real slowdown week open several distinct incidents in
    quick succession, too noisy for per-incident paging.
12. **Capability silence ceiling** (CRITICAL, added as WARN; promoted and tightened 168h→120h
    2026-09-21) — backstop for capabilities excluded from #6 because `silence_grace_hours IS
    NULL` (irregular cadence, currently just `sev2-insights`); fires only past a much longer
    fixed ceiling (`silence_ceiling_hours`, 120h for `sev2-insights`). Promoted because
    WARN-forever was itself judged a false-negative risk for a permanent-dark event, even
    though (unlike #11) it has not yet fired on a real occurrence — see `threshold_basis` for
    the historical false-positive-rate analysis behind the 120h number.
13. **Nightly baseline staleness** (CRITICAL, added as WARN; promoted 2026-09-21, evidence-
    based) — `response_field_baseline` (the nightly job schema_field_missing's comparison
    depends on) hasn't recomputed in >36h. Promoted on direct evidence: job-run history shows
    a real 4-run/~3-day nightly outage 2026-09-04..07 that this threshold would have caught.
14. **Unacknowledged critical** (CRITICAL, promoted 2026-09-21, promote-on-principle) —
    backstop rollup: any CRITICAL incident still neither acknowledged nor resolved. Was
    already labeled CRITICAL in code but had never been wired into `obs_incidents`, so it
    never actually persisted or notified — this promotion fixes that gap rather than tuning a
    threshold. Not empirically validated (`obs_incidents` had 0 rows at promotion time); FP
    risk is structurally bounded since it can only fire on top of an already-real incident.
    Scope tightened 2026-09-22 (noise review): excludes whichever detector(s) are currently
    digest-routed and adds a 1h age floor — without this, a live incident showed this rollup
    firing a second, un-digested Teams card for the same finding a digest-routed detector had
    already posted, within seconds. Generalized same day (auto fan-out routing follow-up) from
    a hardcoded 3-name list to `digest_routed_detectors_sql()` (cell 2) — any detector whose
    open incident count exceeds `FANOUT_DIGEST_THRESHOLD` within `FANOUT_WINDOW_MINUTES`, with
    those incidents clustered within `FANOUT_CLUSTER_MINUTES` of each other, is routed
    automatically, not just the 3 detectors known to need it as of this writing.
15. **Long running incident** (CRITICAL, promoted 2026-09-21, promote-on-principle) —
    age-based backstop: any incident open 3+ days unacknowledged. Same unwired-CRITICAL bug
    and same promote-on-principle basis as #14.
16. **ETL row count anomaly** (CRITICAL, added 2026-09-21, bronze-projection gap review) — an
    ETL run reported `status='success'` on schedule and took a normal amount of time, but
    `rows_written` broke that table's all-time-consistent zero/nonzero regime — a silent
    partial/no-op load. Closes a blind spot #8/#10/#11 all share: none of them look at row
    counts. Live-data confirmed: across ~45,000 historical runs the pattern is cleanly bimodal
    (14 tables always write >0 rows, 5 tables always write exactly 0, zero crossovers ever) —
    this is the first-ever crossing, not an arbitrary threshold. Shipped CRITICAL and
    digest-routed from day one (see `ptof_obs_liveness_detection.ipynb`) since its fan-out
    shape mirrors #10's real 14/19-table simultaneous event.

## Retired detectors (2026-09-21 detector value audit)
- **Write lag anomalies** — `write_lag_s` has been exactly 0 for all 10,721 rows over 30 days:
  zero variance, so the MAD-based bound was degenerate and untunable regardless of floor. Not
  a threshold problem — the signal carried no information in prod. Table and check() removed
  entirely rather than left WARN-forever with nothing to promote. See
  `ptof_obs_liveness_detection.ipynb` — a fresh detector can be added later if write-latency
  instrumentation is ever wired up for real.
- **Schema drift (field_added half)** — 0 historical occurrences ever, and semantically a new
  field appearing isn't a failure mode the way a field disappearing is. `schema_field_missing`
  (#5 above) stays CRITICAL and covers the actual risk; `response_schema_drift` table itself
  is untouched.

## Severity semantics
`raise` fires **only on UNAVAILABLE**, so task status means *"is monitoring working"* rather
than *"did it find something"*. CRITICAL findings route to Teams and `obs_incidents`.

## Notification policy
A CRITICAL incident notifies when first detected, then not again for 24 h unless still
unacknowledged. Broken detectors notify every run. There are currently no WARN-status
detectors in this pipeline (see above) — the WARN/no-notify path exists in code only for any
future provisional detector. Any detector currently digest-routed (see
`digest_routed_detectors_sql()`, cell 2 — as of this writing that's `etl_run_slow`,
`etl_table_staleness`, `etl_row_count_anomaly`, but membership is computed per run from live,
clustered fan-out, not fixed) is excluded from per-incident notify and routed to its own daily
digest instead — see cell 11.

## Acknowledgement
Two ways to acknowledge, both landing on the same `obs_incidents.acknowledged_at`/
`acknowledged_by` columns:

- **Per-detector query** (in each detector's card section, cell 10) —
  `WHERE detector='{detector}' AND acknowledged_at IS NULL`. Detector-scoped, not incident-
  scoped: it acknowledges every currently-unacknowledged incident under that detector, including
  ones not rendered on the current card (e.g. already notified within the last 24h and so
  excluded from this run's incident list).
- **Mass-acknowledge query** (added 2026-09-22, noise/friction review; header of the card,
  cell 10, only rendered when a card has more than one incident) —
  `WHERE acknowledged_at IS NULL AND (detector, source_row_id) IN (...)`, built from the exact
  `(detector, source_row_id)` pairs in the in-memory `incidents` list that produced this
  specific card. Scoped precisely to what was just read, not a broader time/severity window —
  it can never acknowledge something the developer didn't actually see in this card. Added so
  clearing a multi-detector alert is one paste instead of one query per detector section.

Acknowledgement does not survive re-detection: `WHEN MATCHED` (cell 5) resets
`acknowledged_at`/`acknowledged_by` to `NULL` on every run that still matches, so acknowledging
an incident only "sticks" once its detector stops matching (at which point auto-resolve closes
it on the same condition, independent of ack state). The real re-page throttle during a
sustained incident is `notified_at`'s 24h cooldown above, not `acknowledged_at`.

## Thresholds
Recorded with their basis in `mq_gmdf_dev.oil_obs.threshold_basis`. `etl_pipeline_staleness`'s
30-minute global threshold was reviewed and explicitly left unchanged (2026-09-21, FP/FN bias
review step 3, signed off "skip") — a 28min retune was considered and declined as not worth the
churn for the marginal false-negative-window reduction.

## Digest fan-out routing
`digest_routed_detectors_sql()` (cell 2) replaced a hardcoded 3-detector allowlist (2026-09-22,
auto fan-out routing follow-up to the noise review above). Any detector with more than
`FANOUT_DIGEST_THRESHOLD` (currently 5) open incidents touched within the trailing
`FANOUT_WINDOW_MINUTES` (currently 60), **and** whose oldest and newest qualifying
`first_detected` fall within `FANOUT_CLUSTER_MINUTES` (currently 15) of each other, is routed
to a bundled once-per-24h Teams digest instead of a per-incident page — both cell 4's
`unacknowledged_critical` exclusion and cell 11's per-incident notify/digest logic call this
one function, so they cannot drift out of sync the way the old duplicated list already did
once (dropped from cell 4 by a merge, restored in commit 1369b54). The clustering condition
(added 2026-09-22, same-day re-review) exists because the count threshold alone can't tell a
genuine fan-out burst (many rows appearing together, like both real events below) apart from
an unrelated backlog that slowly crosses the same count — which would otherwise get bundled
into a digest *and* disappear from `unacknowledged_critical`, hiding exactly the kind of
neglected backlog that rollup exists to catch. All three constants are provisional, chosen
below the two real fan-out events on record (5 correlated `etl_run_slow` runs/week, 14/19
tables in `etl_table_staleness`, both single bursts) — revisit once there's a real per-run
fan-out distribution to look at.

## Prod source migration (2026-09-10)
Reads from prod source tables via views (`v_llm_bronze` → `mq_gmdf_dp_prd.oil.ptof_primary__ai_shift_outputs`,
`v_ish_bronze` → `mq_gmdf_dp_prd.oil.ptof_ish_audit`, `v_etl_bronze` → `mq_gmdf_dp_prd.oil.ptof_etl_pipeline_audit`).
All writes stay in `mq_gmdf_dev.oil_obs`. 7 detectors dropped (latency, error rate, hallucination,
transport violations, prompt size, credential fastfail, rapid human correction).

In [ ]:
WEBHOOK = dbutils.secrets.get(scope="obs-alerting", key="teams-webhook")
CAT = "mq_gmdf_dev.oil_obs"

print(f"webhook_configured={bool(WEBHOOK)}")

In [ ]:
import json
import re
from datetime import datetime, timezone

results = []  # name, severity, value, detail


def check(name, sql, severity_fn, detail_fn=None):
    """Run a scalar-ish check. Missing table -> UNAVAILABLE (a finding, not a crash)."""
    try:
        rows = spark.sql(sql).collect()
    except Exception as e:  # noqa: BLE001 - any failure should surface as a finding
        results.append({"name": name, "severity": "UNAVAILABLE", "value": None,
                        "detail": str(e).split("\n")[0][:300]})
        return
    if not rows:
        results.append({"name": name, "severity": "OK", "value": 0, "detail": "no rows"})
        return
    row = rows[0].asDict()
    detail = detail_fn(row) if detail_fn else json.dumps(
        {k: (str(v) if v is not None else None) for k, v in row.items()})
    results.append({"name": name, "severity": severity_fn(row), "value": row, "detail": detail})


# gt0: every check() call in this notebook uses this severity function. There is no WARN-status
# detector left in this pipeline (2026-09-21 detector value audit) -- a matching "warn0" lambda
# was removed here (dead code, never called) for the same reason. If a future detector needs to
# ship WARN-only/provisional, add a similarly-named lambda back at that point.
gt0   = lambda r: "CRITICAL" if (r.get("n") or 0) > 0 else "OK"  # noqa: E731

# Fan-out auto-routing to digest (2026-09-22, noise-review follow-up). Replaces a hardcoded
# 3-detector allowlist (etl_run_slow / etl_table_staleness / etl_row_count_anomaly) that was
# used identically in two places -- cell 4's unacknowledged_critical check and cell 11's
# per-incident notify query -- and had already drifted out of sync once (a merge silently
# dropped one of the two copies; see cell 4's "RESTORED 2026-09-22" note). Both consumers now
# call this single function instead of each hardcoding their own copy of the list, so there is
# exactly one place to edit and no way for a future merge to desync two independently-worded
# clauses -- a merge conflict on this function is loud, not a silent partial drop.
#
# Basis for "fan-out" as the routing signal, not a fixed list: two real detectors have needed
# digest treatment so far (5 correlated etl_run_slow slowdowns/week; 14/19 tables stale in
# etl_table_staleness), both discovered only after the fact by a human noticing the pattern.
# This makes that discovery automatic instead of requiring another manual promotion next time.
#
# Threshold is provisional, same caveat as the promote-on-principle detectors below: chosen
# comfortably under the two real fan-out events on record (5/day, 14 tables) without being so
# low that a normal small cluster (2-3 related rows) gets bounced into digest mode and loses
# per-incident visibility. Revisit once there's a real per-run fan-out distribution to look at.
FANOUT_DIGEST_THRESHOLD = 5
FANOUT_WINDOW_MINUTES = 60

# FANOUT_CLUSTER_MINUTES (2026-09-22, re-review follow-up): a raw open-incident count can't
# distinguish one correlated fan-out event (etl_run_slow/etl_table_staleness -- many rows that
# all appeared together) from an unrelated backlog that slowly crossed the same count (several
# distinct, genuinely separate findings on one detector that each just failed to get resolved).
# The latter deserves per-incident visibility, not a bundled digest -- and since
# unacknowledged_critical excludes whatever this function returns, misclassifying it would also
# hide that backlog from the one rollup meant to catch neglected incidents. Requiring the open
# rows' first_detected timestamps to cluster within this window is a direct measurement of
# "appeared together", matching both real fan-out events on record (both were a single burst,
# not a slow accumulation). Provisional like the two constants above -- revisit together.
FANOUT_CLUSTER_MINUTES = 15


def digest_routed_detectors_sql(as_of="current_timestamp()", table=None):
    """Subquery: detectors currently fanning out enough to route to digest, not per-incident.

    Reads persisted obs_incidents, not in-memory state, so cell 4 (which runs before this
    run's MERGE) and cell 11 (which runs after) each see the freshest state available to them
    at their point in the run -- the same one-cycle lag cell 4's other checks already have
    against this run's own detections, not a new inconsistency.

    Requires the qualifying rows' first_detected to fall within FANOUT_CLUSTER_MINUTES of each
    other, not just a bare count over FANOUT_WINDOW_MINUTES -- a count threshold alone can't
    tell a genuine fan-out burst apart from an unrelated backlog that happened to cross the
    same number slowly.

    as_of/table default to current production behavior (now, real obs_incidents) -- extended
    2026-09-22 (backtest support) so ptof_obs_backtest_replay.ipynb can point this at
    obs_incidents_backtest and a stepped historical timestamp without duplicating this query.
    """
    table = table or f"{CAT}.obs_incidents"
    return f"""
        SELECT detector FROM {table}
        WHERE resolved_at IS NULL
          AND last_detected >= {as_of} - INTERVAL {FANOUT_WINDOW_MINUTES} MINUTES
        GROUP BY detector
        HAVING count(*) > {FANOUT_DIGEST_THRESHOLD}
           AND cast(max(first_detected) as long) - cast(min(first_detected) as long)
               <= {FANOUT_CLUSTER_MINUTES} * 60
    """

## Persist findings to obs_incidents

Runs after the scalar checks in the cell above (so their results are in `results` for
`SCALAR_INCIDENT_SOURCES` to read) and before the table-backed liveness/output/handover checks
further down. `unacknowledged_critical`/`long_running_incident` (moved up into the scalar-check
cell 2026-09-21 as part of their promotion) therefore see the *previous* run's persisted
`obs_incidents` state, not this run's -- a one job-cycle (~5-10 min) lag against a 24h notify
window / 3-day age threshold that doesn't change what they catch.

MERGE on `(detector, source_row_id)` means re-detecting increments `detection_count` rather
than duplicating, and acknowledgement survives across runs.

Any `SKIPPED` line is a column-name mismatch in `INCIDENT_SOURCES` — fix the list rather
than ignoring it, or that detector's findings are never persisted.

**Re-detection clears `resolved_at` and `acknowledged_at`:** `WHEN MATCHED` sets both to `NULL`
in addition to refreshing `last_detected`/`detection_count`. Without this, a detector whose key
is stable across occurrences (a constant literal id, or `capability`+`field`/`config` with no
per-event discriminator — `blank_output`, `schema_field_missing`, `handover_delivery_rate`,
`pipeline_heartbeat`, `etl_pipeline_staleness`) would go permanently dark the first time it is
resolved-or-acknowledged and later recurs: the notify query, `unacknowledged_critical`, and
auto-resolve itself all filter on both columns being NULL, so a frozen non-null value on either
one silently suppresses every future occurrence with no error. A fresh re-detection is a new
occurrence — an old acknowledgment of a since-resolved instance shouldn't silence it. Detectors
keyed on a genuinely per-event id (`handover_delivery` on `ish_row_id`, `etl_pipeline_failure`
on `run_id`) were never exposed to this, since a resolved occurrence can't recur under the same
id.

**Auto-resolve:** immediately after the MERGE loop, any incident whose detector ran cleanly
this run but did not re-detect it gets `resolved_at` stamped -- the condition stopped
recurring. This is the only place `resolved_at` is set automatically outside of re-detection;
`acknowledged_by`/`acknowledged_at` are otherwise hand-set only (cleared on re-detection above,
never set automatically). A detector that errored this run is excluded from auto-resolve for
its own incidents (see `_skipped_detectors`), so a broken query can never be misread as "the
condition went away."

In [ ]:
# pipeline_heartbeat and etl_pipeline_staleness are scalar CRITICAL checks with no natural
# per-row grain (unlike blank_output/schema_drift/etc, which have a findings table) -- just one
# global condition each. Run them here, before incident persistence, so cell below can MERGE
# their result into obs_incidents this same run instead of only printing to the job log.
#
# Threshold tightened 2h -> 45min -> 25min (2026-09-21, FP/FN bias review step 3, priority 1,
# signed off): real max gap in v_llm_bronze over 30 days is 16.4 min (still true as of this
# retune); 25 min keeps a 1.5x false-positive margin over that observed max while cutting the
# false-negative exposure window a further ~44% vs. the 45min interim value.
#
# Refactored 2026-09-22 (backtest support) into def <detector>_sql(as_of) builders so
# ptof_obs_backtest_thresholds.ipynb / ptof_obs_backtest_replay.ipynb can evaluate each check at
# any historical point -- every check() call below still runs with the default
# as_of="current_timestamp()", so production behavior is unchanged.
def pipeline_heartbeat_sql(as_of="current_timestamp()"):
    return f"""
    SELECT CASE WHEN count(*) = 0 THEN 1 ELSE 0 END AS n, count(*) AS rows_last_25m
    FROM {CAT}.v_llm_bronze
    WHERE called_at >= {as_of} - INTERVAL 25 MINUTES
    """

check("pipeline_heartbeat", pipeline_heartbeat_sql(), gt0)


def etl_pipeline_staleness_sql(as_of="current_timestamp()"):
    return f"""SELECT CASE WHEN max(run_timestamp) < {as_of} - INTERVAL 30 MINUTES
                    THEN 1 ELSE 0 END AS n,
           max(run_timestamp) AS latest_run
        FROM {CAT}.v_etl_bronze"""

check("etl_pipeline_staleness", etl_pipeline_staleness_sql(), gt0)

# nightly_baseline_staleness (promoted WARN -> CRITICAL 2026-09-21, detector value audit,
# evidence-based): if the nightly baseline job fails silently, schema drift runs against stale
# thresholds. 36h gives ~12h grace past the nightly schedule. Promotion basis: job-run history
# shows a real 4-run/~3-day nightly outage 2026-09-04..07 that a 36h threshold would have
# caught -- direct evidence, not a principle-only promotion. Moved here (from its old spot in
# cell 7) so it runs before the incident-persistence cell below and can be picked up by
# SCALAR_INCIDENT_SOURCES the same way pipeline_heartbeat/etl_pipeline_staleness are.
def nightly_baseline_staleness_sql(as_of="current_timestamp()"):
    return f"""SELECT count(*) AS n, max(computed_at) AS last_computed
        FROM {CAT}.response_field_baseline
        WHERE computed_at < {as_of} - INTERVAL 36 HOURS"""

check("nightly_baseline_staleness", nightly_baseline_staleness_sql(), gt0)

# unacknowledged_critical / long_running_incident (promoted WARN -> CRITICAL 2026-09-21,
# promote-on-principle, NOT empirically validated -- obs_incidents has 0 rows as of this
# promotion, so there is no historical false-positive data either way): both are backstop
# rollups over obs_incidents itself, and both were already labeled/intended as CRITICAL-grade
# signals but had never been wired into INCIDENT_SOURCES/SCALAR_INCIDENT_SOURCES, so neither
# ever persisted or notified despite the label -- this promotion fixes that gap rather than
# tuning a threshold. False-positive risk is structurally bounded: each can only fire on top
# of an already-real, already-CRITICAL incident that itself passed its own detector's bar.
#
# Moved here (from cell 7) to slot into the same before-persistence pattern as the checks
# above. Trade-off: they now see the PREVIOUS run's persisted obs_incidents state rather than
# this run's (the reverse of the old ordering note in cell 3, now updated) -- one job-cycle
# (~5-10 min) of staleness against a 24h notify window / 3-day age threshold is immaterial.
#
# unacknowledged_critical WHERE clause tightened 2026-09-22 (noise review -- two real Teams
# cards from the same alert run showed etl_run_slow's own finding AND this rollup both firing
# for the identical incident within the same post): excludes whatever detector(s) are
# currently digest-routed, and adds a 1h floor for every other detector so this rollup answers
# "has this sat unacknowledged for a while" instead of "was this just detected."
#
# GENERALIZED 2026-09-22 (auto fan-out routing follow-up): the digest-detector exclusion below
# was originally a hardcoded 3-name list duplicated in this cell and in cell 11 -- and had
# already drifted out of sync once (a merge dropped this cell's copy while leaving cell 11's
# intact; restored in commit 1369b54). Both cells now call the single digest_routed_detectors_sql()
# function defined in cell 2 instead of maintaining their own copy of the list, so that class of
# regression can no longer happen silently: there is one function to edit, and a merge conflict
# on it is loud rather than a quiet partial drop.
#
# unacknowledged_critical/long_running_incident both take an optional `table` param (default
# real obs_incidents) in addition to `as_of` -- these two operate on obs_incidents itself, not a
# bronze view, so ptof_obs_backtest_replay.ipynb points `table` at obs_incidents_backtest for
# its lifecycle replay. ptof_obs_backtest_thresholds.ipynb skips both entirely: there is no
# bronze-only historical signal to sweep for a detector whose input is the incident table.
def unacknowledged_critical_sql(as_of="current_timestamp()", table=None):
    table = table or f"{CAT}.obs_incidents"
    return f"""
    SELECT count(*) AS n,
           concat_ws(', ', collect_set(concat(detector,':',coalesce(capability,'-')))) AS detail,
           min(first_detected) AS oldest
    FROM {table}
    WHERE severity = 'CRITICAL'
      AND acknowledged_at IS NULL
      AND resolved_at IS NULL
      AND detector NOT IN ({digest_routed_detectors_sql(as_of=as_of, table=table)})
      AND first_detected <= {as_of} - INTERVAL 1 HOUR
    """

check("unacknowledged_critical", unacknowledged_critical_sql(), gt0)


def long_running_incident_sql(as_of="current_timestamp()", table=None):
    table = table or f"{CAT}.obs_incidents"
    return f"""
    SELECT count(*) AS n,
           concat_ws(', ', collect_set(concat(detector,' open ',
                      cast(datediff({as_of}, first_detected) AS STRING),'d'))) AS detail
    FROM {table}
    WHERE resolved_at IS NULL
      AND acknowledged_at IS NULL
      AND first_detected <= {as_of} - INTERVAL 3 DAYS
    """

check("long_running_incident", long_running_incident_sql(), gt0)

In [ ]:
INCIDENT_SOURCES = [
    # detector, source table, id column, capability column, severity, payload columns, extra WHERE
    #
    # 6 active detectors that persist to obs_incidents:
    ("handover_delivery",  "handover_delivery_failures", "ish_row_id",
     None,                  "CRITICAL",
     ["failure_reason", "shift_date", "batch_nbr"], ""),
    ("blank_output", "blank_output_findings", "finding_signature",
     "capability", "CRITICAL",
     ["model_config", "blank_rate_window", "blank_count_window", "total_calls_window",
      "latest_hour"], ""),
    ("schema_field_missing", "response_schema_drift", "finding_signature",
     "capability", "CRITICAL",
     ["model_configs_seen", "field_name", "drift_type",
      "baseline_presence_rate", "current_present", "current_rows"],
     "WHERE drift_type = 'field_missing'"),
    ("handover_delivery_rate", "handover_delivery_rate", "'handover_delivery_rate_global'",
     None,                     "CRITICAL",
     ["failure_pct_7d", "sent_ok", "failed", "last_attempt"],
     # Absolute-count OR-trigger added 2026-09-21 (FP/FN bias review, priority 4, signed off):
     # the rate-only path needs (sent_ok + failed) >= 10 to evaluate at all, which structurally
     # can't fire during a low-volume week (current pace ~14 attempts/7d) no matter how bad a
     # small failure cluster is. `failed >= 3` is a hard floor independent of sample size.
     "WHERE (failure_pct_7d > 20 AND (sent_ok + failed) >= 10) OR failed >= 3"),
    ("etl_pipeline_failure", "etl_pipeline_health", "finding_signature",
     None, "CRITICAL",
     ["table_or_view", "status", "error_message", "run_timestamp", "duration_seconds"], ""),
    # etl_table_staleness (added 2026-09-21): per-table companion to the etl_pipeline_staleness
    # scalar below -- table-backed (finding_signature is a constant sha2(table_or_view, 256))
    # so it fits the standard table-backed MERGE loop rather than SCALAR_INCIDENT_SOURCES.
    ("etl_table_staleness", "etl_table_staleness", "finding_signature",
     None, "CRITICAL",
     ["table_or_view", "minutes_since_last_run", "last_run_at"], ""),
    # etl_run_slow (promoted WARN -> CRITICAL 2026-09-21, FP/FN bias review, priority 2, signed
    # off): 5 real, correlated, multi-table slowdown events observed in ~1 week while WARN/
    # log-only -- a proven-real signal that was invisible to a human. finding_signature is
    # sha2(task_name || window_start), so table-backed like the detectors above. Still persists
    # here for tracking/auto-resolve; per-incident Teams notification is suppressed in favor of
    # the daily digest below (step 3 follow-up, signed off "should be changed to a digest").
    ("etl_run_slow", "etl_run_slow", "finding_signature",
     None, "CRITICAL",
     ["task_name", "anomalous_count", "affected_table_count", "affected_tables",
      "max_duration_s", "upper_bound_s"], ""),
    # capability_silence_ceiling (promoted WARN -> CRITICAL and tightened 168h -> 120h
    # 2026-09-21, FP/FN bias review follow-up, signed off: "ok lets do it"): backstop for
    # capabilities excluded from capability_silence (WARN, checked in cell 7) because
    # silence_grace_hours IS NULL -- currently just sev2-insights. WARN-forever was judged a
    # false-negative risk in its own right for a permanent-dark event, even though this
    # detector (unlike etl_run_slow) has not yet fired on a real occurrence. finding_signature
    # is a constant sha2(capability, 256), same lifecycle pattern as etl_table_staleness.
    ("capability_silence_ceiling", "capability_silence_ceiling", "finding_signature",
     "capability", "CRITICAL",
     ["silence_ceiling_hours", "hours_since_last_call", "last_call_at", "owner"], ""),
    # shift_context_missing (promoted WARN -> CRITICAL 2026-09-21, FP/FN bias review step 3,
    # signed off): standing data-quality gap that silently degrades every AI-to-ISH correlation
    # join -- judged not worth leaving invisible to a human indefinitely. finding_signature is a
    # constant sha2(capability, 256), same lifecycle pattern as capability_silence_ceiling.
    ("shift_context_missing", "shift_context_missing", "finding_signature",
     "capability", "CRITICAL",
     ["total_calls", "blank_shift_type", "blank_batch_nbr", "null_shift_date", "last_seen"], ""),
    # capability_silence (promoted WARN -> CRITICAL 2026-09-21, detector value audit,
    # evidence-based): full historical call-gap distribution for the 3 covered capabilities
    # shows 0 empirical false positives ever, with real margin over the grace threshold
    # (saa-display/situational-awareness 7.4x, summary 3.0x) -- see
    # ptof_obs_liveness_detection.ipynb for the query. finding_signature is a constant
    # sha2(capability, 256), same lifecycle pattern as capability_silence_ceiling.
    ("capability_silence", "capability_silence", "finding_signature",
     "capability", "CRITICAL",
     ["expected_min_daily", "silence_grace_hours", "owner", "calls_last_7d",
      "last_call_at", "hours_since_last_call"], ""),
    # etl_row_count_anomaly (added 2026-09-21, bronze-projection gap review, live-data
    # confirmed): closes a blind spot etl_pipeline_failure/etl_table_staleness/etl_run_slow all
    # share -- none of them look at rows_written, so a run can report status='success' on
    # schedule and still silently write zero rows (or the reverse). finding_signature is a
    # constant sha2(table_or_view, 256), same lifecycle pattern as etl_table_staleness. Digest-
    # routed from day one, not per-incident -- see cell 11 -- since its fan-out shape mirrors
    # etl_table_staleness's real 14/19-table simultaneous event.
    ("etl_row_count_anomaly", "etl_row_count_anomaly", "finding_signature",
     None, "CRITICAL",
     ["table_or_view", "rows_written", "regime", "run_timestamp"], ""),
]

# pipeline_heartbeat and etl_pipeline_staleness have no findings table (see the check() calls
# above) -- just a single global scalar condition each. Persisted below as constant-id rows
# straight from that check() result, so they get the same obs_incidents lifecycle (dedup,
# notify/re-notify, auto-resolve) as the table-backed detectors above.
#
# nightly_baseline_staleness/unacknowledged_critical/long_running_incident (promoted WARN ->
# CRITICAL 2026-09-21) added below -- their check() calls moved into cell 4 above so they run
# before this cell and land in `results` the same way pipeline_heartbeat/etl_pipeline_staleness
# already do.
SCALAR_INCIDENT_SOURCES = [
    ("pipeline_heartbeat", "pipeline_heartbeat_global"),
    ("etl_pipeline_staleness", "etl_staleness_global"),
    ("nightly_baseline_staleness", "nightly_baseline_staleness_global"),
    ("unacknowledged_critical", "unacknowledged_critical_global"),
    ("long_running_incident", "long_running_incident_global"),
]

# BACKTRACK: "Where to look" queries for the Teams card. Point at the durable bronze views
# (append-only) rather than detector findings tables (CREATE OR REPLACE'd over rolling windows).
# All lineage strings reference prod source tables via the views.
def _sqlq(v):
    return str(v).replace("'", "''") if v is not None else ""

BACKTRACK = {
    "handover_delivery": {
        "table": "v_ish_bronze",
        "where": lambda cap, p, row_id, last_detected: f"id = '{_sqlq(row_id)}'",
        "lineage": "ptof_obs_behavioral_correlation.ipynb -> handover_delivery_failures -> "
                   "v_ish_bronze -> mq_gmdf_dp_prd.oil.ptof_ish_audit",
    },
    "handover_delivery_rate": {
        "table": "v_ish_bronze",
        "lineage": "ptof_obs_behavioral_correlation.ipynb -> handover_delivery_rate -> "
                   "v_ish_bronze -> mq_gmdf_dp_prd.oil.ptof_ish_audit",
        "where": lambda cap, p, row_id, last_detected: (
            "entity_type = 'HandoverEmail' AND ts BETWEEN "
            f"TIMESTAMP'{last_detected}' - INTERVAL 7 DAYS AND TIMESTAMP'{last_detected}'"),
        "order_by": "ts DESC",
    },
    "blank_output": {
        "table": "v_llm_bronze",
        "lineage": "ptof_obs_mal_output.ipynb -> blank_output_findings -> "
                   "v_llm_bronze -> mq_gmdf_dp_prd.oil.ptof_primary__ai_shift_outputs",
        "where": lambda cap, p, row_id, last_detected: (
            f"capability = '{_sqlq(cap)}' AND is_blank_output = true AND called_at BETWEEN "
            f"TIMESTAMP'{last_detected}' - INTERVAL 6 HOURS AND TIMESTAMP'{last_detected}'"),
        "order_by": "called_at DESC",
    },
    "schema_field_missing": {
        "table": "v_llm_bronze",
        "lineage": "ptof_obs_mal_output.ipynb -> response_schema_drift -> "
                   "v_llm_bronze -> mq_gmdf_dp_prd.oil.ptof_primary__ai_shift_outputs",
        "where": lambda cap, p, row_id, last_detected: (
            f"capability = '{_sqlq(cap)}' AND called_at BETWEEN "
            f"TIMESTAMP'{last_detected}' - INTERVAL 24 HOURS AND TIMESTAMP'{last_detected}'"),
        "order_by": "called_at DESC",
        "note": "Field-missing can't be expressed as a plain filter — inspect response_parsed "
                "on these rows directly.",
    },
    "etl_pipeline_failure": {
        "table": "v_etl_bronze",
        "lineage": "ptof_obs_liveness_detection.ipynb -> etl_pipeline_health -> "
                   "v_etl_bronze -> mq_gmdf_dp_prd.oil.ptof_etl_pipeline_audit",
        "where": lambda cap, p, row_id, last_detected: (
            f"table_or_view = '{_sqlq(p.get('table_or_view', ''))}' AND run_timestamp BETWEEN "
            f"TIMESTAMP'{last_detected}' - INTERVAL 24 HOURS AND TIMESTAMP'{last_detected}'"),
        "order_by": "run_timestamp DESC",
    },
    "etl_table_staleness": {
        "table": "v_etl_bronze",
        "lineage": "ptof_obs_liveness_detection.ipynb -> etl_table_staleness -> "
                   "v_etl_bronze -> mq_gmdf_dp_prd.oil.ptof_etl_pipeline_audit",
        "where": lambda cap, p, row_id, last_detected: (
            f"table_or_view = '{_sqlq(p.get('table_or_view', ''))}' AND run_timestamp BETWEEN "
            f"TIMESTAMP'{last_detected}' - INTERVAL 24 HOURS AND TIMESTAMP'{last_detected}'"),
        "order_by": "run_timestamp DESC",
        "note": "Scoped to this one table only -- if etl_pipeline_staleness (the global check) "
                "is quiet, that does NOT mean this is a false alarm; it only fires when EVERY "
                "table stops.",
    },
    "etl_run_slow": {
        "table": "v_etl_bronze",
        "lineage": "ptof_obs_liveness_detection.ipynb -> etl_run_slow -> "
                   "v_etl_bronze -> mq_gmdf_dp_prd.oil.ptof_etl_pipeline_audit",
        "where": lambda cap, p, row_id, last_detected: (
            f"task_name = '{_sqlq(p.get('task_name', ''))}' AND run_timestamp BETWEEN "
            f"TIMESTAMP'{last_detected}' - INTERVAL 24 HOURS AND TIMESTAMP'{last_detected}'"),
        "order_by": "run_timestamp DESC",
        "note": "Rolled up to task_name -- affected_tables in the payload lists which of that "
                "task's tables were slow this occurrence; all of them typically move together.",
    },
    "etl_row_count_anomaly": {
        "table": "v_etl_bronze",
        "lineage": "ptof_obs_liveness_detection.ipynb -> etl_row_count_anomaly -> "
                   "v_etl_bronze -> mq_gmdf_dp_prd.oil.ptof_etl_pipeline_audit",
        "where": lambda cap, p, row_id, last_detected: (
            f"table_or_view = '{_sqlq(p.get('table_or_view', ''))}' AND run_timestamp BETWEEN "
            f"TIMESTAMP'{last_detected}' - INTERVAL 24 HOURS AND TIMESTAMP'{last_detected}'"),
        "order_by": "run_timestamp DESC",
        "note": "regime in the payload shows which all-time-consistent zero/nonzero pattern "
                "this table just broke -- rows_written on the offending run is the value that "
                "crossed it.",
    },
    "capability_silence_ceiling": {
        "table": "v_llm_bronze",
        "lineage": "ptof_obs_liveness_detection.ipynb -> capability_silence_ceiling -> "
                   "v_llm_bronze -> mq_gmdf_dp_prd.oil.ptof_primary__ai_shift_outputs",
        "where": lambda cap, p, row_id, last_detected: (
            f"capability = '{_sqlq(cap)}' AND called_at BETWEEN "
            f"TIMESTAMP'{last_detected}' - INTERVAL 30 DAYS AND TIMESTAMP'{last_detected}'"),
        "order_by": "called_at DESC",
        "note": "Expect zero or very sparse rows -- that's the condition. Window widened to "
                "30 days here since this capability's normal cadence is irregular.",
    },
    "shift_context_missing": {
        "table": "v_llm_bronze",
        "lineage": "ptof_obs_liveness_detection.ipynb -> shift_context_missing -> "
                   "v_llm_bronze -> mq_gmdf_dp_prd.oil.ptof_primary__ai_shift_outputs",
        "where": lambda cap, p, row_id, last_detected: (
            f"capability = '{_sqlq(cap)}' AND called_at BETWEEN "
            f"TIMESTAMP'{last_detected}' - INTERVAL 7 DAYS AND TIMESTAMP'{last_detected}'"),
        "order_by": "called_at DESC",
        "note": "Rows returned include ones with populated shift context too -- filter further "
                "on shift_type/batch_nbr/shift_date being blank or null to isolate the gap.",
    },
    "capability_silence": {
        "table": "v_llm_bronze",
        "lineage": "ptof_obs_liveness_detection.ipynb -> capability_silence -> "
                   "v_llm_bronze -> mq_gmdf_dp_prd.oil.ptof_primary__ai_shift_outputs",
        "where": lambda cap, p, row_id, last_detected: (
            f"capability = '{_sqlq(cap)}' AND called_at BETWEEN "
            f"TIMESTAMP'{last_detected}' - INTERVAL 7 DAYS AND TIMESTAMP'{last_detected}'"),
        "order_by": "called_at DESC",
    },
    "pipeline_heartbeat": {
        "table": "v_llm_bronze",
        "lineage": "ptof_obs_alert.ipynb -> pipeline_heartbeat -> "
                   "v_llm_bronze -> mq_gmdf_dp_prd.oil.ptof_primary__ai_shift_outputs",
        "where": lambda cap, p, row_id, last_detected: (
            "called_at BETWEEN "
            f"TIMESTAMP'{last_detected}' - INTERVAL 6 HOURS AND TIMESTAMP'{last_detected}'"),
        "order_by": "called_at DESC",
        "note": "Expect zero rows in the trailing 25min — that's the condition. Window widened "
                "to 6h here to show the most recent activity before it stopped.",
    },
    "etl_pipeline_staleness": {
        "table": "v_etl_bronze",
        "lineage": "ptof_obs_alert.ipynb -> etl_pipeline_staleness -> "
                   "v_etl_bronze -> mq_gmdf_dp_prd.oil.ptof_etl_pipeline_audit",
        "where": lambda cap, p, row_id, last_detected: (
            "run_timestamp BETWEEN "
            f"TIMESTAMP'{last_detected}' - INTERVAL 6 HOURS AND TIMESTAMP'{last_detected}'"),
        "order_by": "run_timestamp DESC",
    },
    "nightly_baseline_staleness": {
        "table": "response_field_baseline",
        "lineage": "ptof_obs_nightly_baseline.ipynb -> response_field_baseline",
        "where": lambda cap, p, row_id, last_detected: "1=1",
        "order_by": "computed_at DESC",
        "note": "Global check -- inspect the whole table, not scoped to a capability. "
                "Promoted 2026-09-21 on direct evidence: job-run history shows a real "
                "4-run/~3-day nightly outage 2026-09-04..07 that this threshold would catch.",
    },
    "unacknowledged_critical": {
        "table": "obs_incidents",
        "lineage": "ptof_obs_alert.ipynb -> obs_incidents (self-referential meta-check)",
        "where": lambda cap, p, row_id, last_detected: (
            "severity = 'CRITICAL' AND acknowledged_at IS NULL AND resolved_at IS NULL"),
        "order_by": "first_detected ASC",
        "note": "Promoted on principle 2026-09-21, not empirical evidence -- obs_incidents had "
                "0 rows at promotion time. Was already labeled CRITICAL but never wired into "
                "INCIDENT_SOURCES, so it never actually notified; this fixes that gap.",
    },
    "long_running_incident": {
        "table": "obs_incidents",
        "lineage": "ptof_obs_alert.ipynb -> obs_incidents (self-referential meta-check)",
        "where": lambda cap, p, row_id, last_detected: (
            "resolved_at IS NULL AND acknowledged_at IS NULL AND "
            "first_detected <= current_timestamp() - INTERVAL 3 DAYS"),
        "order_by": "first_detected ASC",
        "note": "Promoted on principle 2026-09-21, not empirical evidence -- obs_incidents had "
                "0 rows at promotion time. Bounded FP risk: can only fire on top of an "
                "already-real, already-CRITICAL incident.",
    },
}

# Timestamp before the emission loop so auto-resolve can distinguish "ran and found nothing"
# from "hasn't run yet". Detectors that error are tracked in _skipped_detectors and excluded
# from auto-resolve.
_incident_run_ts = spark.sql("SELECT current_timestamp() AS ts").first().ts
_skipped_detectors = set()

for detector, table, id_col, cap_col, severity, payload_cols, extra in INCIDENT_SOURCES:
    try:
        cap_expr = cap_col if cap_col else "CAST(NULL AS STRING)"
        payload = ", ".join(f"'{c}', CAST({c} AS STRING)" for c in payload_cols)
        source_expr = f"{CAT}.{table}"
        spark.sql(f"""
            MERGE INTO {CAT}.obs_incidents t
            USING (
              SELECT '{detector}'             AS detector,
                     CAST({id_col} AS STRING) AS source_row_id,
                     {cap_expr}               AS capability,
                     '{severity}'             AS severity,
                     to_json(map({payload}))  AS signal_payload
              FROM {source_expr} {extra}
            ) s
            ON t.detector = s.detector AND t.source_row_id = s.source_row_id
            WHEN MATCHED THEN UPDATE SET
                t.severity        = s.severity,
                t.last_detected   = current_timestamp(),
                t.detection_count = t.detection_count + 1,
                t.signal_payload  = s.signal_payload,
                t.resolved_at     = NULL,
                t.acknowledged_at = NULL,
                t.acknowledged_by = NULL
            WHEN NOT MATCHED THEN INSERT
                (detector, source_row_id, capability, severity,
                 first_detected, last_detected, detection_count, signal_payload)
              VALUES
                (s.detector, s.source_row_id, s.capability, s.severity,
                 current_timestamp(), current_timestamp(), 1, s.signal_payload)
        """)
        print(f"  emitted: {detector}")
    except Exception as e:  # noqa: BLE001
        _skipped_detectors.add(detector)
        print(f"  SKIPPED {detector}: {str(e).split(chr(10))[0][:160]}")

# Scalar checks: MERGE a synthetic constant-id row only when the check() result (computed
# above, before this cell) came back CRITICAL -- skipping the MERGE otherwise has the same
# effect as "no matching rows" for a table-backed detector, so auto-resolve (below) still
# clears a resolved condition without any extra code.
for detector, source_row_id in SCALAR_INCIDENT_SOURCES:
    result = next((r for r in results if r["name"] == detector), None)
    if result is None or result["severity"] == "UNAVAILABLE":
        _skipped_detectors.add(detector)
        print(f"  SKIPPED {detector}: check did not run cleanly")
        continue
    if result["severity"] != "CRITICAL":
        continue
    try:
        payload = json.dumps({k: str(v) for k, v in result["value"].items() if k != "n"})
        spark.sql(f"""
            MERGE INTO {CAT}.obs_incidents t
            USING (
              SELECT '{detector}' AS detector, '{source_row_id}' AS source_row_id,
                     CAST(NULL AS STRING) AS capability, 'CRITICAL' AS severity,
                     '{payload.replace("'", "''")}' AS signal_payload
            ) s
            ON t.detector = s.detector AND t.source_row_id = s.source_row_id
            WHEN MATCHED THEN UPDATE SET
                t.severity        = s.severity,
                t.last_detected   = current_timestamp(),
                t.detection_count = t.detection_count + 1,
                t.signal_payload  = s.signal_payload,
                t.resolved_at     = NULL,
                t.acknowledged_at = NULL,
                t.acknowledged_by = NULL
            WHEN NOT MATCHED THEN INSERT
                (detector, source_row_id, capability, severity,
                 first_detected, last_detected, detection_count, signal_payload)
              VALUES
                (s.detector, s.source_row_id, s.capability, s.severity,
                 current_timestamp(), current_timestamp(), 1, s.signal_payload)
        """)
        print(f"  emitted: {detector}")
    except Exception as e:  # noqa: BLE001
        _skipped_detectors.add(detector)
        print(f"  SKIPPED {detector}: {str(e).split(chr(10))[0][:160]}")

print("Incident emission complete.")

In [ ]:
# Auto-resolve: an incident whose detector ran cleanly this run but didn't re-MERGE it means
# the underlying condition stopped recurring -- last_detected stays behind _incident_run_ts.
# Without this, resolved_at is "set by hand" only, so the open/unacknowledged backlog (235
# CRITICAL incidents observed live, some untouched since first detection) grows unbounded
# regardless of whether the condition is still real. Runs across both acknowledged and
# unacknowledged rows -- resolution reflects the condition's state, not human review status.
#
# Detectors in _skipped_detectors are excluded: a broken query must never be read as "the
# condition went away" (that would silently clear real incidents the moment their detector broke).
_active_detectors = ([d for d, *_ in INCIDENT_SOURCES if d not in _skipped_detectors]
                      + [d for d, _ in SCALAR_INCIDENT_SOURCES if d not in _skipped_detectors])
if _active_detectors:
    _detector_list = ", ".join(f"'{d}'" for d in _active_detectors)
    spark.sql(f"""
        UPDATE {CAT}.obs_incidents
        SET resolved_at = current_timestamp()
        WHERE resolved_at IS NULL
          AND detector IN ({_detector_list})
          AND last_detected < '{_incident_run_ts}'
    """)
    print(f"Auto-resolve checked {len(_active_detectors)} detector(s) that ran cleanly this run.")
else:
    print("No detectors ran cleanly this run; skipping auto-resolve.")

In [ ]:
# === Liveness checks ===

# capability_silence_ceiling (added 2026-09-21, item #2; promoted WARN -> CRITICAL and
# tightened 168h -> 120h 2026-09-21, FP/FN bias review follow-up, signed off) is no longer
# checked here -- it's table-backed with a finding_signature now and persists via the standard
# INCIDENT_SOURCES MERGE loop above (cell 5), same as etl_table_staleness/etl_run_slow.

# shift_context_missing (promoted WARN -> CRITICAL 2026-09-21, FP/FN bias review step 3,
# signed off) is no longer checked here -- it's table-backed with a finding_signature now and
# persists via the standard INCIDENT_SOURCES MERGE loop above (cell 5), same as
# capability_silence_ceiling/etl_table_staleness/etl_run_slow.

# capability_silence (promoted WARN -> CRITICAL 2026-09-21, detector value audit, evidence-
# based) is no longer checked here -- it's table-backed with a finding_signature now and
# persists via the standard INCIDENT_SOURCES MERGE loop above (cell 5), same as
# shift_context_missing/capability_silence_ceiling.

# write_lag_anomalies RETIRED 2026-09-21 (detector value audit): write_lag_s has been exactly 0
# for all 10,721 rows over 30 days -- zero variance, so the MAD-based bound is degenerate and
# untunable no matter the floor. Not a threshold problem; the signal itself carries no
# information in prod. The underlying table and this check() have both been removed rather than
# left running WARN-forever with nothing to promote. If write latency instrumentation is later
# wired up for real (a genuinely nonzero, varying write_lag_s), a fresh detector can be added
# then against real data -- see ptof_obs_liveness_detection.ipynb.

# === ETL health checks ===

# etl_pipeline_failure: any ETL task failure in the last 24h. The agent continues producing
# outputs on stale data — capability_silence and pipeline_heartbeat won't fire.
check(
    "etl_pipeline_failure",
    f"""SELECT count(*) AS n,
           concat_ws(', ', collect_set(table_or_view)) AS failed_tables
        FROM {CAT}.etl_pipeline_health""",
    gt0,
)

# etl_table_staleness (added 2026-09-21): per-table companion to etl_pipeline_staleness above --
# catches a partial ETL stall masked by the global max(run_timestamp) staying fresh off
# unaffected tables. See HANDOFF.md for the supplement-vs-replace reasoning.
check(
    "etl_table_staleness",
    f"""SELECT count(*) AS n,
           concat_ws(', ', collect_set(table_or_view)) AS stale_tables
        FROM {CAT}.etl_table_staleness""",
    gt0,
)

# etl_run_slow (added 2026-09-18; grain changed to task_name/window_start 2026-09-21; promoted
# WARN -> CRITICAL 2026-09-21, FP/FN bias review priority 2, signed off): ETL run duration
# degraded beyond its own MAD-based historical bound, clustered (>=3 in an hour) across however
# many tables share that task_name. 5 real, correlated, multi-table slowdown events were
# observed in ~1 week while this was WARN/log-only and invisible to a human -- promoted so it
# now persists to obs_incidents and posts to Teams like the other CRITICALs above.
check(
    "etl_run_slow",
    f"""
    SELECT count(*) AS n,
           concat_ws(', ', collect_set(concat(task_name,'=',
                      cast(anomalous_count AS STRING),' slow runs across ',
                      cast(affected_table_count AS STRING),' table(s)'))) AS detail
    FROM {CAT}.etl_run_slow
    """,
    gt0,
)

# etl_row_count_anomaly (added 2026-09-21, bronze-projection gap review, live-data confirmed):
# an ETL run reported status='success' on schedule but rows_written broke that table's
# all-time-consistent zero/nonzero regime -- a silent partial/no-op load that
# etl_pipeline_failure/etl_table_staleness/etl_run_slow all miss since none of them look at
# rows_written. Table-backed with a constant finding_signature, same lifecycle as
# etl_table_staleness. Per-incident Teams notification is suppressed in favor of a daily digest
# (see cell 11) -- its fan-out shape mirrors etl_table_staleness's real 14/19-table event.
check(
    "etl_row_count_anomaly",
    f"""SELECT count(*) AS n,
           concat_ws(', ', collect_set(concat(table_or_view,' (',regime,')'))) AS detail
        FROM {CAT}.etl_row_count_anomaly""",
    gt0,
)

# === Output quality checks ===

# blank_output: response content empty on output records. Invisible to every other detector.
check(
    "blank_output",
    f"""
    SELECT count(*) AS n, max(blank_rate_window) AS worst_rate
    FROM {CAT}.blank_output_findings
    """,
    gt0,
)

# schema_drift (field_added half) RETIRED 2026-09-21 (detector value audit): 0 historical
# occurrences ever, and semantically a new field appearing is not a failure mode the way a field
# disappearing is -- schema_field_missing (below) stays CRITICAL and covers the actual risk.
# response_schema_drift table itself is untouched (schema_field_missing still reads it, filtered
# to drift_type = 'field_missing'); only the separate field_added-inclusive check() here is gone.

# schema_field_missing: a field that was reliably present has disappeared. The real risk is
# downstream code reading a missing field as null/false instead of unknown.
check(
    "schema_field_missing",
    f"""
    SELECT count(*) AS n,
           concat_ws(', ', collect_set(concat(capability,': ',field_name))) AS detail
    FROM {CAT}.response_schema_drift
    WHERE drift_type = 'field_missing'
    """,
    gt0,
)

# nightly_baseline_staleness (promoted WARN -> CRITICAL 2026-09-21, detector value audit,
# evidence-based) is no longer checked here -- its check() call moved up into cell 4 so it can
# feed SCALAR_INCIDENT_SOURCES in cell 5, the same pattern pipeline_heartbeat/
# etl_pipeline_staleness already use.

# === Handover delivery checks ===

# handover_delivery_rate: fires if delivery gets worse than the ~9.3% baseline (rate path,
# needs >=10 attempts to be meaningful) OR an absolute-count floor is hit regardless of sample
# size (added 2026-09-21, FP/FN bias review priority 4, signed off) -- current volume is only
# ~14 attempts/7d, so the rate path alone could never fire on a real failure cluster in a
# low-volume week.
check(
    "handover_delivery_rate",
    f"""
    SELECT failure_pct_7d, sent_ok, failed, last_attempt,
           CASE WHEN (failure_pct_7d > 20 AND (sent_ok + failed) >= 10) OR failed >= 3
                THEN 1 ELSE 0 END AS n
    FROM {CAT}.handover_delivery_rate
    """,
    gt0,
)

# handover_delivery_new_failure REMOVED (2026-09-21, dead-code audit): this check() ran and
# logged to the job output, but was never added to INCIDENT_SOURCES/SCALAR_INCIDENT_SOURCES,
# so it never persisted to obs_incidents or reached Teams -- pure dead weight. Also fully
# redundant with handover_delivery above, which already persists every unacknowledged failure
# per-row (keyed on ish_row_id) and notifies on it.

# === Incident state checks ===

# unacknowledged_critical / long_running_incident (promoted WARN -> CRITICAL 2026-09-21,
# promote-on-principle -- fixes a pre-existing bug where both were labeled CRITICAL but never
# wired into INCIDENT_SOURCES/SCALAR_INCIDENT_SOURCES, so neither ever notified) are no longer
# checked here -- both moved up into cell 4 so they can feed SCALAR_INCIDENT_SOURCES in cell 5,
# the same pattern pipeline_heartbeat/etl_pipeline_staleness already use.

In [ ]:
order = {"CRITICAL": 0, "UNAVAILABLE": 1, "WARN": 2, "OK": 3}
results.sort(key=lambda r: order.get(r["severity"], 9))

for r in results:
    print(f"{r['severity']:<12} {r['name']:<40} {r['detail']}")

criticals   = [r for r in results if r["severity"] == "CRITICAL"]
unavailable = [r for r in results if r["severity"] == "UNAVAILABLE"]


In [ ]:
DETECTOR_META = {
    "handover_delivery": {
        "label": "Shift handover email failed to send",
        "what": "An automated shift handover never reached PFS3_ISH_SME@lists.lilly.com. "
                "The audit row exists; the email does not.",
        "triage": [
            "getaddrinfo failure = DNS resolution to the mail host, not an app bug",
            "Check whether the affected shift was handed over by other means",
            "Baseline is ~9.3% of send attempts since Jun 19 — check handover_delivery_rate to "
            "see whether this is drift or the standing rate",
        ],
    },
    "blank_output": {
        "label": "Agent returned a blank response",
        "what": "An output record exists but its content was empty. Any blank output at any "
                "volume is a finding (2026-09-21, Option A signed off) — invisible to every "
                "other detector in this system.",
        "triage": [
            "Check whether this is a prompt/template regression or an upstream input problem",
            "blank_rate_window / blank_count_window / total_calls_window show how bad and how big",
            "A blank handover or summary reaching a real person is worse than an error — nobody "
            "else is watching for this",
        ],
    },
    "schema_field_missing": {
        "label": "Expected output field went missing",
        "what": "A field that was reliably present in this capability's response schema has "
                "disappeared. Downstream automation reading a missing field as null/false "
                "instead of unknown is the real risk here.",
        "triage": [
            "Check for a recent prompt-template edit or model swap on this capability",
            "baseline_presence_rate tells you how reliably the field used to appear",
            "All 4 prod capabilities produce grounded narratives with predictable output shapes",
        ],
    },
    "handover_delivery_rate": {
        "label": "Shift handover delivery rate deteriorating",
        "what": "Handover email failures over the trailing 7 days have exceeded the established "
                "baseline, with enough attempts (>=10) for the rate to be meaningful.",
        "triage": [
            "Compare against the ~9.3% historical baseline noted under handover_delivery — "
            "this fires only once it gets worse than that",
            "Check handover_delivery_failures for the underlying getaddrinfo/DNS pattern",
            "sent_ok / failed counts are in the payload below",
        ],
    },
    "etl_pipeline_failure": {
        "label": "ETL source data pipeline failure",
        "what": "A source table refresh in the ETL pipeline failed. The SAA agent "
                "may be running on stale data — outputs look normal but the underlying "
                "alarms, interventions, or cycle status are not current.",
        "triage": [
            "Check table_or_view in the payload — which source table failed?",
            "Check ptof_etl_pipeline_audit for the error_message and duration_seconds",
            "If the ETL is stuck, the agent's outputs are stale but still arriving — "
            "capability_silence and pipeline_heartbeat will NOT fire",
        ],
    },
    "pipeline_heartbeat": {
        "label": "No agent outputs in 25 minutes — total outage",
        "what": "Zero rows have landed in v_llm_bronze across ALL capabilities for 25 minutes "
                "straight. This is the backstop for a total upstream outage -- every other "
                "detector implicitly assumes outputs keep arriving.",
        "triage": [
            "Check whether ptof_primary__ai_shift_outputs is receiving writes at all",
            "rows_last_25m in the payload below confirms the zero",
            "capability_silence fires per-capability on a longer grace window; this fires "
            "regardless of capability once ingestion itself stops",
        ],
    },
    "etl_pipeline_staleness": {
        "label": "ETL hasn't completed a run in 30 minutes",
        "what": "The upstream ETL refresh (~19 tables every 10-15 min) has gone quiet for "
                "30+ minutes. Outputs keep arriving (so pipeline_heartbeat/capability_silence "
                "stay quiet) but the agent is reasoning over stale source data.",
        "triage": [
            "latest_run in the payload below is the last completed ETL timestamp",
            "Check whether the ETL job/scheduler itself is stuck or failed to even start a run",
            "etl_pipeline_failure needs an actual failed run recorded -- this fires on the "
            "absence of any run, which that one can't see",
            "etl_table_staleness (per-table) can be firing even when this one is quiet -- this "
            "check only trips when EVERY table stops, not just one",
        ],
    },
    "etl_table_staleness": {
        "label": "ETL source table hasn't run in over an hour",
        "what": "A single ETL source table has gone quiet for 60+ minutes even though other "
                "tables in the fleet are still refreshing on schedule. Per-table companion to "
                "etl_pipeline_staleness -- catches a partial stall that the global check's "
                "fleet-wide max(run_timestamp) would otherwise mask (confirmed gap: the "
                "2026-09-19/20 weekend incident, 14 of 19 tables stalled ~25h while 5 kept the "
                "global max fresh).",
        "triage": [
            "table_or_view in the payload identifies exactly which source stopped",
            "Check whether that table's upstream job/task is stuck, failed silently, or was "
            "descheduled -- last_run_at / minutes_since_last_run show how long it's been out",
            "etl_pipeline_staleness (global) staying quiet does NOT mean this is a false alarm "
            "-- that check only fires when EVERY table stops, this one is scoped to just this "
            "table",
        ],
    },
    "etl_run_slow": {
        "label": "ETL run duration degraded across a task's tables",
        "what": "ETL run duration for this task_name degraded beyond its own MAD-based "
                "historical bound, clustered (>=3 slow runs in an hour). Promoted from WARN "
                "2026-09-21 after 5 real, correlated, multi-table slowdown events were observed "
                "in ~1 week while it was log-only and invisible to a human.",
        "triage": [
            "task_name in the payload identifies the task; affected_tables lists which of its "
            "tables were slow this occurrence -- they typically move together",
            "max_duration_s vs upper_bound_s shows how far past the historical bound this run got",
            "Check whether this correlates with an etl_pipeline_failure or etl_table_staleness "
            "finding around the same window -- a slowdown can precede an outright stall",
        ],
    },
    "etl_row_count_anomaly": {
        "label": "ETL run wrote an unexpected row count",
        "what": "An ETL run reported status='success' on schedule and took a normal amount of "
                "time, but rows_written broke this table's all-time-consistent zero/nonzero "
                "regime -- a silent partial or no-op load. Added 2026-09-21 after live data "
                "across ~45,000 historical runs showed the pattern is cleanly bimodal (14 "
                "source tables always write >0 rows, 5 materialized-view/gate tables always "
                "write exactly 0, zero crossovers ever) -- this is the first-ever crossing, "
                "not an arbitrary row-count threshold. Closes a blind spot shared by "
                "etl_pipeline_failure/etl_table_staleness/etl_run_slow, none of which look at "
                "rows_written.",
        "triage": [
            "regime in the payload shows which all-time pattern (expect_zero / expect_nonzero) "
            "this table just broke; rows_written is the value that crossed it",
            "expect_nonzero -> 0 usually means a silent partial/failed load that still reported "
            "success -- check for an upstream filter or join that dropped every row",
            "expect_zero -> nonzero on a materialized-view/gate table means something wrote "
            "data where none was expected -- check for a logic change in that refresh job",
            "Delivered as a daily digest, not per-incident (see the digest note in cell 11) -- "
            "its fan-out shape mirrors etl_table_staleness's real 14/19-table event",
        ],
    },
    "capability_silence_ceiling": {
        "label": "Capability has gone silent past its long-cadence ceiling",
        "what": "A capability with an irregular natural cadence (currently only sev2-insights, "
                "structurally excluded from the shorter-grace capability_silence check) hasn't "
                "produced any output in longer than its silence_ceiling_hours backstop. "
                "Promoted from WARN 2026-09-21 -- WARN-forever was itself judged a false-"
                "negative risk for a permanent-dark event, even though this detector has not "
                "yet fired on a real occurrence (unlike etl_run_slow before its promotion).",
        "triage": [
            "hours_since_last_call vs silence_ceiling_hours in the payload shows how far past "
            "the ceiling this is",
            "Check with the capability's owner (in the payload) whether this is a genuine "
            "outage or a legitimate cadence shift that means the ceiling itself needs revisiting",
            "capability_silence (the shorter-grace check) does not cover this capability at "
            "all -- this is the only detection path for it going dark",
        ],
    },
    "shift_context_missing": {
        "label": "Shift context fields missing from output records",
        "what": "Output records for this capability are landing with blank/null shift_type, "
                "batch_nbr, or shift_date. These fields are what enables per-shift and "
                "per-batch slicing and AI-to-ISH correlation joins -- a gap here silently "
                "degrades every downstream join without erroring. Promoted from WARN "
                "2026-09-21 -- a standing data-quality gap judged not worth leaving invisible "
                "to a human indefinitely, on the same promote-on-principle basis as "
                "capability_silence_ceiling (not a fresh empirical FP re-validation).",
        "triage": [
            "blank_shift_type / blank_batch_nbr / null_shift_date in the payload show which "
            "field(s) are gapping and how often, out of total_calls",
            "Check whether this capability's upstream caller is passing shift context at all, "
            "or whether it's a parsing/mapping gap introduced downstream",
            "last_seen shows the most recent affected call -- if it predates a recent fix, the "
            "condition may already be resolved and will auto-clear on the next clean run",
        ],
    },
    "capability_silence": {
        "label": "Capability has gone silent past its grace window",
        "what": "A registered capability (saa-display, situational-awareness, or summary) "
                "hasn't produced any output within its own silence_grace_hours window. "
                "Promoted from WARN 2026-09-21 on real historical evidence: full call-gap "
                "distribution for all 3 covered capabilities shows 0 empirical false positives "
                "ever, with real margin over the grace threshold (saa-display/situational-"
                "awareness 7.4x, summary 3.0x).",
        "triage": [
            "hours_since_last_call vs silence_grace_hours in the payload shows how far past "
            "grace this is",
            "calls_last_7d / last_call_at show recent volume and the exact last output",
            "Check with the capability's owner (in the payload) whether this is a genuine "
            "outage upstream or a legitimate cadence change",
            "sev2-insights is not covered here (silence_grace_hours IS NULL) -- see "
            "capability_silence_ceiling for that capability's backstop",
        ],
    },
    "nightly_baseline_staleness": {
        "label": "Nightly response-field baseline hasn't refreshed",
        "what": "response_field_baseline (the nightly job that schema_field_missing's "
                "comparison baseline depends on) hasn't recomputed in over 36 hours. If this "
                "job fails silently, schema drift keeps comparing against a stale baseline "
                "instead of catching real drift. Promoted from WARN 2026-09-21 on direct "
                "evidence: job-run history shows a real 4-run/~3-day nightly outage "
                "2026-09-04..07 that this 36h threshold would have caught.",
        "triage": [
            "last_computed in the payload is the most recent baseline refresh timestamp",
            "Check the ptof_obs_nightly_baseline job's run history for failures around that time",
            "While this is open, treat schema_field_missing findings with extra caution -- its "
            "baseline may be out of date",
        ],
    },
    "unacknowledged_critical": {
        "label": "Backlog of unacknowledged CRITICAL incidents",
        "what": "A rollup across every detector: CRITICAL incidents that are still neither "
                "acknowledged nor resolved. Promoted from WARN 2026-09-21 -- this was already "
                "labeled CRITICAL in code but had never been wired into obs_incidents, so it "
                "never actually persisted or notified despite the label. Promoted on principle, "
                "not fresh empirical evidence (obs_incidents had 0 rows at promotion time); "
                "false-positive risk is structurally bounded since this can only fire on top "
                "of an already-real CRITICAL incident.",
        "triage": [
            "detail in the payload lists which detector(s)/capability(ies) make up the backlog",
            "oldest shows when the earliest of these was first detected",
            "Acknowledge or resolve the underlying incidents directly -- this rollup clears "
            "itself once they do",
        ],
    },
    "long_running_incident": {
        "label": "Incident open 3+ days with no acknowledgement",
        "what": "An age-based backstop (not a recurrence-count one): any incident that has sat "
                "unresolved and unacknowledged for 3+ days. Promoted from WARN 2026-09-21 on "
                "the same basis as unacknowledged_critical -- already labeled CRITICAL but "
                "never wired to persist/notify. Promoted on principle, not fresh empirical "
                "evidence; bounded FP risk since it can only fire on top of an already-real "
                "incident.",
        "triage": [
            "detail in the payload lists which incident(s) and how many days each has been open",
            "A 3-day-old unacknowledged CRITICAL usually means the original Teams notification "
            "was missed, not that the condition resolved itself -- check the underlying "
            "detector's current state directly",
        ],
    },
}


def _age(first_detected):
    """Human-readable age."""
    delta = datetime.now(timezone.utc) - first_detected.replace(tzinfo=timezone.utc)
    d, h = delta.days, delta.seconds // 3600
    if d:
        return f"{d}d {h}h old"
    m = (delta.seconds % 3600) // 60
    return f"{h}h {m}m old" if h else f"{m}m old"


def _job_run_url():
    """Link back to this job run. Absent in interactive runs, which is fine."""
    try:
        ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        host = ctx.tags().get("browserHostName").get()
        job_id = ctx.tags().get("jobId").get()
        run_id = ctx.tags().get("multitaskParentRunId").get()
        return f"https://{host}/jobs/{job_id}/runs/{run_id}"
    except Exception:  # noqa: BLE001
        return None


_EMAIL_LOCAL_PART_RE = re.compile(r"([^,;\s@]+)@")


def _mask_emails(s):
    """Defense-in-depth: mask the local-part of any email before it can reach a Teams card."""
    return _EMAIL_LOCAL_PART_RE.sub("***@", s)


def _facts_from_payload(payload):
    """signal_payload as labelled facts rather than a raw JSON blob."""
    try:
        d = json.loads(payload) if payload else {}
    except Exception:  # noqa: BLE001
        return [{"title": "payload", "value": str(payload)[:200]}]
    pretty = {
        "failure_reason": "Reason", "shift_date": "Shift date", "batch_nbr": "Batch",
        "model_config": "Model config", "model_configs_seen": "Model configs seen",
        "failure_pct_7d": "Failure % (7d)", "sent_ok": "Sent OK", "failed": "Failed",
        "last_attempt": "Last attempt",
        "blank_rate_window": "Blank rate", "blank_count_window": "Blank count",
        "total_calls_window": "Total calls (window)", "latest_hour": "Latest hour",
        "field_name": "Field", "drift_type": "Drift type",
        "baseline_presence_rate": "Baseline presence rate", "current_present": "Currently present",
        "current_rows": "Current rows",
        "table_or_view": "Table/view", "status": "Status", "error_message": "Error message",
        "run_timestamp": "Run timestamp", "duration_seconds": "Duration (s)",
        "rows_last_25m": "Rows last 25m", "latest_run": "Latest ETL run",
        "minutes_since_last_run": "Minutes since last run", "last_run_at": "Last run at",
        "task_name": "Task", "anomalous_count": "Anomalous run count",
        "affected_table_count": "Affected table count", "affected_tables": "Affected tables",
        "max_duration_s": "Max duration (s)", "upper_bound_s": "Upper bound (s)",
        "rows_written": "Rows written", "regime": "Expected regime",
        "silence_ceiling_hours": "Silence ceiling (h)",
        "hours_since_last_call": "Hours since last call", "last_call_at": "Last call at",
        "owner": "Owner",
        "total_calls": "Total calls", "blank_shift_type": "Blank shift type",
        "blank_batch_nbr": "Blank batch #", "null_shift_date": "Null shift date",
        "last_seen": "Last seen",
        "expected_min_daily": "Expected min daily", "silence_grace_hours": "Silence grace (h)",
        "calls_last_7d": "Calls (7d)", "last_computed": "Last computed baseline",
        "detail": "Detail", "oldest": "Oldest",
    }
    return [{"title": pretty.get(k, k), "value": _mask_emails(str(v))[:180]}
            for k, v in d.items() if v not in (None, "", "null")]

In [ ]:
def post_teams(incidents, broken):
    """POST a triage-oriented Adaptive Card to the Teams webhook (Workflows-based, expect 202).

    try/except on purpose: a Teams outage must not fail the alert task, or a notification
    problem becomes indistinguishable from a monitoring problem.
    """
    if not WEBHOOK:
        print("No webhook configured; skipping notification.")
        return
    import requests
    import time

    severity = "DETECTOR BROKEN" if broken else "CRITICAL"
    headline = (f"{len(broken)} detector(s) stopped working" if broken
                else f"{len(incidents)} critical finding(s)")

    body = [{
        "type": "Container", "style": "attention", "bleed": True,
        "items": [
            {"type": "TextBlock", "size": "Large", "weight": "Bolder",
             "text": f"{severity} — {headline}"},
            {"type": "TextBlock", "spacing": "None", "isSubtle": True, "wrap": True,
             "text": f"ISH agent observability · "
                     f"{datetime.now(timezone.utc):%Y-%m-%d %H:%M} UTC"},
        ],
    }]

    # Mass-acknowledge (2026-09-22, noise/friction review): one UPDATE covering every incident
    # actually rendered in THIS card, so a dev clearing a multi-detector alert doesn't have to
    # copy-paste one query per detector section below. Built from the exact `incidents` list
    # passed in here -- not a fresh WHERE severity='CRITICAL' scan -- so it can never silently
    # ack something outside what was just read: no time-window race, no scope drift. Skipped
    # for single-incident cards since the per-detector query below is already the same query.
    if len(incidents) > 1:
        pairs = ", ".join(
            f"('{_sqlq(d)}', '{_sqlq(sid)}')" for d, sid in
            {(r["detector"], r["source_row_id"]) for r in incidents}
        )
        body.append({
            "type": "Container", "separator": True, "spacing": "Medium",
            "items": [
                {"type": "TextBlock", "weight": "Bolder", "size": "Small",
                 "text": f"Acknowledge all {len(incidents)} incidents in this alert"},
                {"type": "TextBlock", "wrap": True, "spacing": "None", "isSubtle": True,
                 "text": "Only affects exactly what's shown below — nothing detected after "
                         "this card was sent."},
                {"type": "TextBlock", "wrap": True, "spacing": "None", "fontType": "Monospace",
                 "text": f"UPDATE {CAT}.obs_incidents SET acknowledged_by=current_user(), "
                         f"acknowledged_at=current_timestamp() WHERE acknowledged_at IS NULL "
                         f"AND (detector, source_row_id) IN ({pairs});"},
            ],
        })

    # Broken detectors first: monitoring has stopped, which outranks anything it found.
    for r in broken:
        body.append({
            "type": "Container", "style": "attention", "separator": True,
            "items": [
                {"type": "TextBlock", "weight": "Bolder", "wrap": True,
                 "text": f"Detector stopped: {r['name']}"},
                {"type": "TextBlock", "wrap": True, "isSubtle": True,
                 "text": "This check could not run — a table is missing or a query threw. "
                         "Nothing is watching this condition until it is fixed."},
                {"type": "TextBlock", "wrap": True, "fontType": "Monospace",
                 "text": r["detail"][:400]},
            ],
        })

    # Group by detector so multiple findings of one kind read as one problem.
    by_detector = {}
    for r in incidents:
        by_detector.setdefault(r["detector"], []).append(r)

    for detector, rows in by_detector.items():
        meta = DETECTOR_META.get(detector, {})
        oldest = min(r["first_detected"] for r in rows)
        caps = sorted({x["capability"] for x in rows if x["capability"]})

        items = [
            {"type": "TextBlock", "size": "Medium", "weight": "Bolder", "wrap": True,
             "text": meta.get("label", detector)},
            {"type": "TextBlock", "spacing": "None", "isSubtle": True, "wrap": True,
             "text": f"`{detector}`"
                     + (f" · {', '.join(caps)}" if caps else "")
                     + f" · {len(rows)} occurrence(s) · oldest {_age(oldest)}"},
        ]
        if meta.get("what"):
            items.append({"type": "TextBlock", "wrap": True, "text": meta["what"]})

        newest = max(rows, key=lambda x: x["first_detected"])
        facts = _facts_from_payload(newest["signal_payload"])
        if facts:
            items.append({"type": "TextBlock", "weight": "Bolder", "size": "Small",
                          "spacing": "Medium",
                          "text": "Most recent occurrence" if len(rows) > 1 else "Details"})
            items.append({"type": "FactSet", "facts": facts})

        # BACKTRACK: a copy-paste query against the durable bronze view, not this detector's
        # own findings table (which is CREATE OR REPLACE'd over a rolling window).
        bt = BACKTRACK.get(detector)
        if bt:
            payload_dict = json.loads(newest["signal_payload"]) if newest["signal_payload"] else {}
            where_clause = bt["where"](newest["capability"], payload_dict,
                                       newest["source_row_id"], newest["last_detected"])
            query = f"SELECT * FROM {CAT}.{bt['table']} WHERE {where_clause}"
            if bt.get("order_by"):
                query += f" ORDER BY {bt['order_by']}"
            query += " LIMIT 20;"
            items.append({"type": "TextBlock", "weight": "Bolder", "size": "Small",
                          "spacing": "Medium", "text": "Where to look"})
            if bt.get("lineage"):
                items.append({"type": "TextBlock", "wrap": True, "spacing": "None",
                              "isSubtle": True, "text": bt["lineage"]})
            if bt.get("note"):
                items.append({"type": "TextBlock", "wrap": True, "spacing": "None",
                              "isSubtle": True, "text": bt["note"]})
            items.append({"type": "TextBlock", "wrap": True, "spacing": "None",
                          "fontType": "Monospace", "text": query})

        if meta.get("triage"):
            items.append({"type": "TextBlock", "weight": "Bolder", "size": "Small",
                          "spacing": "Medium", "text": "Where to start"})
            items += [{"type": "TextBlock", "wrap": True, "spacing": "None", "isSubtle": True,
                       "text": f"• {t}"} for t in meta["triage"]]

        items.append({
            "type": "TextBlock", "wrap": True, "spacing": "Medium", "size": "Small",
            "isSubtle": True, "fontType": "Monospace",
            "text": "Acknowledge (suppresses the alert, keeps the record):<br>"
                    f"UPDATE mq_gmdf_dev.oil_obs.obs_incidents SET "
                    f"acknowledged_by=current_user(), acknowledged_at=current_timestamp() "
                    f"WHERE detector='{detector}' AND acknowledged_at IS NULL;",
        })

        body.append({"type": "Container", "separator": True, "spacing": "Medium",
                     "items": items})

    card = {"type": "AdaptiveCard", "version": "1.4", "body": body,
            "$schema": "http://adaptivecards.io/schemas/adaptive-card.json"}

    run_url = _job_run_url()
    if run_url:
        card["actions"] = [{"type": "Action.OpenUrl", "title": "Open job run", "url": run_url}]

    # One bounded retry: a single transient blip shouldn't cost a durable-failure record.
    for attempt in range(2):
        try:
            resp = requests.post(
                WEBHOOK,
                data=json.dumps({"type": "message", "attachments": [{
                    "contentType": "application/vnd.microsoft.card.adaptive",
                    "content": card}]}),
                headers={"Content-Type": "application/json"}, timeout=30)
            print(f"Teams webhook status={resp.status_code}")
            if resp.status_code >= 400:
                print(f"  response: {resp.text[:300]}")
            break
        except Exception as e:  # noqa: BLE001
            print(f"Teams webhook FAILED (attempt {attempt + 1}/2): {str(e)[:200]}")
            if attempt == 0:
                time.sleep(2)
                continue
            try:
                day_key = datetime.now(timezone.utc).strftime("%Y-%m-%d")
                exc_text = str(e)[:500].replace("'", "''")
                spark.sql(f"""
                    MERGE INTO {CAT}.obs_incidents t
                    USING (
                      SELECT 'alerting_pipeline' AS detector,
                             'teams_notify_failed_{day_key}' AS source_row_id,
                             CAST(NULL AS STRING) AS capability,
                             'CRITICAL' AS severity,
                             to_json(map('error', '{exc_text}')) AS signal_payload
                    ) s
                    ON t.detector = s.detector AND t.source_row_id = s.source_row_id
                    WHEN MATCHED THEN UPDATE SET
                        t.last_detected   = current_timestamp(),
                        t.detection_count = t.detection_count + 1,
                        t.signal_payload  = s.signal_payload,
                        t.resolved_at     = NULL,
                        t.acknowledged_at = NULL,
                        t.acknowledged_by = NULL
                    WHEN NOT MATCHED THEN INSERT
                        (detector, source_row_id, capability, severity,
                         first_detected, last_detected, detection_count, signal_payload)
                      VALUES
                        (s.detector, s.source_row_id, s.capability, s.severity,
                         current_timestamp(), current_timestamp(), 1, s.signal_payload)
                """)
            except Exception as merge_e:  # noqa: BLE001
                print(f"  also failed to record durable failure: {str(merge_e)[:200]}")

In [ ]:
# Notify once per incident, not once per run.
#
# Without this, a persistent finding posts every 5 minutes forever — the alert-fatigue failure this
# design exists to prevent. obs_incidents.notified_at carries the state: an incident is reported
# when first seen, then not again for 24 h unless it is still unacknowledged.
#
# WARN findings are deliberately NOT notified -- there are currently none left in this pipeline
# (2026-09-21 detector value audit): every remaining detector is now either CRITICAL (persisted
# + notified) or retired. This mechanism stays in place for any future WARN detector that ships
# provisional/unvalidated.
#
# Digest-routed detectors excluded from per-incident notify (originally etl_run_slow /
# etl_table_staleness / etl_row_count_anomaly, added individually 2026-09-21 after each was
# observed fanning out into many simultaneous incidents -- see cell 2's digest_routed_detectors_sql
# docstring for the two real events that motivated this). GENERALIZED 2026-09-22: rather than
# hardcoding those 3 names here (which had already drifted out of sync once against cell 4's
# copy of the same list), this now calls the single digest_routed_detectors_sql() function so
# any detector that starts fanning out -- not just these three -- is caught automatically and
# routed to its own daily digest below instead of paging once per incident.
try:
    to_notify = spark.sql(f"""
        SELECT detector, source_row_id, capability, severity, first_detected, last_detected,
               detection_count, signal_payload
        FROM {CAT}.obs_incidents
        WHERE severity = 'CRITICAL'
          AND detector NOT IN ({digest_routed_detectors_sql()})
          AND acknowledged_at IS NULL
          AND resolved_at IS NULL
          AND (notified_at IS NULL
               OR notified_at < current_timestamp() - INTERVAL 24 HOURS)
        ORDER BY first_detected
    """).collect()
except Exception as e:  # noqa: BLE001
    print(f"incident notify query failed: {str(e).split(chr(10))[0][:160]}")
    to_notify = []

if to_notify or unavailable:
    post_teams(to_notify, unavailable)
    if to_notify:
        spark.sql(f"""
            UPDATE {CAT}.obs_incidents
            SET notified_at = current_timestamp()
            WHERE severity = 'CRITICAL'
              AND detector NOT IN ({digest_routed_detectors_sql()})
              AND acknowledged_at IS NULL AND resolved_at IS NULL
              AND (notified_at IS NULL
                   OR notified_at < current_timestamp() - INTERVAL 24 HOURS)
        """)
else:
    print("No new or stale incidents to notify.")

# Digest posting (generalized 2026-09-22, auto fan-out routing follow-up): bundle every
# unresolved occurrence from the trailing 24h into one Teams post per digest-routed detector,
# at most once per 24h per detector. Gate is a synthetic marker row
# (detector='{detector}_digest', source_row_id='daily_digest') whose own notified_at tracks the
# last digest send for that detector -- it is immediately re-resolved after each send so it
# never pollutes unacknowledged_critical / long_running_incident, which scan all CRITICAL rows
# regardless of detector.
#
# This one loop replaces three separately copy-pasted digest blocks (etl_run_slow,
# etl_table_staleness, etl_row_count_anomaly) that differed only by detector name -- the
# fan-out detection in cell 2 now decides which detector(s) belong here, rather than a fixed
# list, so a newly-fanning-out detector gets digest treatment without a code change.
try:
    digest_detectors = [r.detector for r in spark.sql(digest_routed_detectors_sql()).collect()]
except Exception as e:  # noqa: BLE001
    print(f"digest-routed detector lookup failed: {str(e).split(chr(10))[0][:160]}")
    digest_detectors = []

if not digest_detectors:
    print("No detector currently over the fan-out threshold; nothing to digest.")

for detector in digest_detectors:
    try:
        digest_rows = spark.sql(f"""
            SELECT detector, source_row_id, capability, severity, first_detected, last_detected,
                   detection_count, signal_payload
            FROM {CAT}.obs_incidents
            WHERE detector = '{detector}'
              AND resolved_at IS NULL
              AND first_detected >= current_timestamp() - INTERVAL 24 HOURS
            ORDER BY first_detected
        """).collect()
    except Exception as e:  # noqa: BLE001
        print(f"{detector} digest query failed: {str(e).split(chr(10))[0][:160]}")
        continue

    if not digest_rows:
        print(f"No {detector} occurrences in trailing 24h; nothing to digest.")
        continue

    digest_marker = f"{detector}_digest"
    digest_due = spark.sql(f"""
        SELECT count(*) = 0 OR max(notified_at) < current_timestamp() - INTERVAL 24 HOURS AS due
        FROM {CAT}.obs_incidents
        WHERE detector = '{digest_marker}' AND source_row_id = 'daily_digest'
    """).first().due

    if not digest_due:
        print(f"{detector} digest suppressed (sent within last 24h); "
              f"{len(digest_rows)} occurrence(s) pending.")
        continue

    post_teams(digest_rows, [])
    spark.sql(f"""
        MERGE INTO {CAT}.obs_incidents t
        USING (
          SELECT '{digest_marker}' AS detector, 'daily_digest' AS source_row_id,
                 CAST(NULL AS STRING) AS capability, 'CRITICAL' AS severity,
                 to_json(map('occurrences_in_window',
                             CAST({len(digest_rows)} AS STRING))) AS signal_payload
        ) s
        ON t.detector = s.detector AND t.source_row_id = s.source_row_id
        WHEN MATCHED THEN UPDATE SET
            t.last_detected   = current_timestamp(),
            t.detection_count = t.detection_count + 1,
            t.signal_payload  = s.signal_payload,
            t.notified_at     = current_timestamp(),
            t.resolved_at     = current_timestamp()
        WHEN NOT MATCHED THEN INSERT
            (detector, source_row_id, capability, severity,
             first_detected, last_detected, detection_count, signal_payload,
             notified_at, resolved_at)
          VALUES
            (s.detector, s.source_row_id, s.capability, s.severity,
             current_timestamp(), current_timestamp(), 1, s.signal_payload,
             current_timestamp(), current_timestamp())
    """)
    print(f"{detector} digest sent: {len(digest_rows)} occurrence(s) in trailing 24h.")

In [ ]:
# Raise ONLY on UNAVAILABLE. Task status means "is monitoring working", not "did it find
# something".
#
# UNAVAILABLE means a table is missing or a query threw: a detector silently stopped monitoring,
# which is the condition that hid every problem in this build. That is worth failing a task over.
# CRITICAL findings are tracked in obs_incidents with acknowledgement state and routed to Teams.
if unavailable:
    raise Exception("ISH observability — DETECTOR BROKEN: " + "; ".join(
        f"{r['name']}={r['severity']}" for r in unavailable))

if criticals:
    print(f"\n{len(criticals)} CRITICAL finding(s) above. Task not failed: findings route via "
          f"obs_incidents and Teams, not task status.")
else:
    print("\nNo CRITICAL findings.")

In [ ]:
# DIAGNOSTIC ONLY -- not part of the automated alert run, no output written anywhere. Run by
# hand when deciding whether schema_field_missing's `current_rows >= 10` floor (fixed,
# ptof_obs_mal_output.ipynb cell 2) needs to become per-capability.
#
# Why: schema_field_missing requires >= 10 non-blank outputs in the trailing 24h before it will
# judge a field missing or not -- below that, it stays silent regardless of what the data shows
# (2026-09-22 re-review of the technical reference's worked example: summary/open_items showed
# a 0.98 -> 0 baseline-vs-current drop suppressed purely by only having 8 current_rows, not
# because the drop wasn't real). That floor is one fixed number for every capability, so it's
# a non-issue for high-volume capabilities (saa-display: 1,010 rows/24h in the same example) but
# could mean the check rarely or never actually runs for a capability whose normal daily volume
# sits at or below 10 -- summary and sev2-insights are the two candidates, per that same example.
#
# This does NOT change anything. It only answers: how often, over the last 30 days, did each of
# these two capabilities actually clear 10 non-blank outputs in a trailing-24h window? If the
# answer is "usually," the floor is fine as-is. If the answer is "rarely," the floor is
# effectively disabling this detector for that capability most days, and a per-capability floor
# (same OR-trigger shape as handover_delivery_rate's absolute-count floor, cell 5 above) is
# worth adding.
_field_missing_floor_diag = spark.sql(f"""
    SELECT capability,
           date(called_at) AS day,
           count(*) AS non_blank_rows_in_24h
    FROM {CAT}.v_llm_bronze
    WHERE capability IN ('summary', 'sev2-insights')
      AND called_at >= current_timestamp() - INTERVAL 30 DAYS
      AND is_blank_output = false
    GROUP BY capability, date(called_at)
    ORDER BY capability, day
""")
_field_missing_floor_diag.createOrReplaceTempView("_field_missing_floor_diag")
_field_missing_floor_diag.show(60, truncate=False)

print("\nDays below the current_rows >= 10 floor, by capability:")
spark.sql("""
    SELECT capability,
           count(*) AS days_observed,
           sum(CASE WHEN non_blank_rows_in_24h < 10 THEN 1 ELSE 0 END) AS days_below_10
    FROM _field_missing_floor_diag
    GROUP BY capability
""").show(truncate=False)
